# Vision Project - Notebook 3
## Training a YOLO26 Object Detector on the Passport Dataset

> **Audience:** AI students and AI engineers continuing the passport project from notebooks 1 & 2.

> **Goal:** Take the split dataset + `data.yaml` and train a YOLO26 detector that finds passport fields. Understand every hyperparameter, every metric, and every line of the output.

---

### Where we are in the pipeline

```
Notebook 1      Annotation        ->  raw_data/{images,labels}
Notebook 2      Data split + yaml ->  dataset/{images,labels}/{train,val} + data.yaml
Notebook 3 *    TRAIN YOLO26      ->  best.pt weights + metrics + plots
Notebook 4      Inference + OCR   ->  Structured passport JSON
```

### Quick model context

We're using **YOLO26**, released by Ultralytics in September 2025 — the latest in the YOLO family. Key things that matter for us:
- **End-to-end NMS-free inference** by default (one-to-one head) — simpler deployment
- **ProgLoss + STAL** specifically improve small-object detection (passport fields *are* small relative to the page)
- **~31% faster CPU inference** than YOLO11 — useful when our backend serves predictions
- **MuSGD optimizer** auto-selected for longer training runs

We'll use `yolo26n.pt` (the **n**ano variant) because: smaller model = less overfitting on 50 images, fast to iterate, easy to deploy.

---

# Part A - Quick Start (the 4 cells that actually train the model)

Run these top-to-bottom. Detailed explanations are in **Part B** below.

### A.1 — Imports

`ultralytics` is the package that ships YOLO26. `Path` is for portable filesystem paths.

In [1]:
from ultralytics import YOLO
from pathlib import Path

### A.2 — Load the pretrained model

The first time this runs, it downloads `yolo26n.pt` (~5 MB) from the Ultralytics GitHub release. After that it's cached.

In [2]:
model = YOLO("yolo26n.pt")

### A.3 — Set project directory (so runs save inside `ai/`, not inside `ai/notebooks/`)

This notebook lives at `ai/notebooks/vision-v3.ipynb`, so `Path.cwd()` is `ai/notebooks/`. We want training outputs at `ai/runs/detect/passport_v1/`, **one level up** from the notebook. Hence `Path.cwd().parent`.

In [3]:
PROJECT_DIR = Path.cwd().parent / "runs" / "detect"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Training runs will be saved under: {PROJECT_DIR.resolve()}")

Training runs will be saved under: C:\Abdullah Ahmed\Ai course\Computer Vision\clause-ade\api\runs\detect


### A.4 — Train

This is the actual training call. On a modern laptop CPU, 100 epochs on 40 images takes roughly 10–30 minutes. On a GPU, a few minutes.

In [4]:
results = model.train(
    data="dataset/data.yaml",
    imgsz=640,
    batch=8,
    epochs=100,
    patience=20,
    augment=True,
    project=str(PROJECT_DIR),
    name="passport_v1",
    exist_ok=True,
)

Ultralytics 8.4.216  Python-3.12.14 torch-2.15.0+cpu CPU (AMD Ryzen 5 4500U with Radeon Graphics)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=True, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=passport_v1, nbs=64, nms=None, opset=None, optimize=False, 

RuntimeError: Dataset 'dataset/data.yaml' error  Dataset 'dataset/data.yaml' images not found, missing path 'C:\Users\Rizwan\Desktop\vision-project\ai\notebooks\dataset\images\val'
Note dataset download directory is 'C:\Abdullah Ahmed\Ai course\Project\datasets'. You can update this in 'C:\Users\HP\AppData\Roaming\Ultralytics\settings.json'

### What just happened

When that cell finishes, you'll find a new folder:

```
ai/runs/detect/passport_v1/
├── weights/
│   ├── best.pt          ← highest val mAP across all epochs (USE THIS for inference)
│   └── last.pt          ← final epoch weights
├── results.png          ← loss curves + mAP curves
├── confusion_matrix.png ← per-class confusion
├── val_batch0_pred.jpg  ← predictions on val images
├── labels.jpg           ← ground-truth box distribution
├── args.yaml            ← every hyperparameter used (reproducibility)
└── results.csv          ← raw per-epoch metrics (open in pandas)
```

Open `results.png` first — it tells you in 30 seconds whether training went well. Then `confusion_matrix.png` shows which classes the model confuses with each other.

---

# Part B — What every line means and why it works

Now the real notebook. Skip this if you just wanted weights; stay if you want to be the person on the team who actually understands what's happening.

## 1. What is "training" a model?

**Training** is an iterative optimization process where:

1. The model takes a batch of images
2. It predicts boxes + classes for each
3. A **loss function** measures how wrong the predictions are vs. the labels you made in notebook 1
4. **Backpropagation** computes how every weight in the network contributed to the error
5. An **optimizer** nudges those weights slightly in the direction that would reduce the error
6. Repeat across all batches → that's one **epoch**
7. Repeat across many epochs → the weights converge to values that produce low loss on new images

For object detection, the loss is actually three losses combined:

| Loss component | What it penalizes |
|---|---|
| `box_loss` | How far off the predicted box coordinates are from the ground truth |
| `cls_loss` | How wrong the predicted class is |
| `dfl_loss` | Distribution-focal-loss: how confidently the box edges are localized |

You'll see all three in the training console, decreasing over epochs. If they plateau or rise, something's wrong (we'll diagnose this later).

## 2. Why YOLO (and why YOLO26 specifically)?

### Why YOLO at all
Object detectors broadly fall into two families:

- **Two-stage** (R-CNN, Faster R-CNN) — first propose regions, then classify each. Very accurate, slow.
- **One-stage** (YOLO, SSD, RetinaNet) — one forward pass produces all boxes and classes. Slightly less accurate at the top of leaderboards, but **fast**, simple, and *usually good enough*.

For document AI in production (a passport scanner running on a phone or a backend API), one-stage detectors are the standard choice. YOLO has been the dominant one-stage family since 2016.

### Why YOLO26 over older versions
| Version | Year | Why we don't pick it for new projects |
|---|---|---|
| YOLOv5 | 2020 | Still works, very widely deployed, but missing modern improvements |
| YOLOv8 | 2023 | Solid, lots of tutorials, but superseded |
| YOLO11 | 2024 | Strong baseline, was previous default |
| **YOLO26** | **2025** | **Current default — faster CPU inference, better small-object detection, NMS-free** |

The architectural changes that matter for *us*:

1. **End-to-end NMS-free inference (one-to-one head).** Older YOLOs predict many overlapping boxes per object and then run Non-Maximum Suppression (NMS) to filter them. NMS is a CPU-side post-processing step — it adds latency and complicates deployment. YOLO26's default head predicts one box per object directly, no NMS needed. Our backend serves passport predictions; one less post-processing step is one less thing to tune.
2. **ProgLoss + Small-Target-Aware Label Assignment (STAL).** Specifically designed to detect small objects better. Passport fields like `dob` occupy maybe 2% of the image — they're "small objects" by detection standards. This is a concrete win for our task.
3. **MuSGD optimizer.** A hybrid of standard SGD and Muon-style orthogonalized updates, automatically selected for longer training runs (>10k iterations). For our 100-epoch run on 40 images that's still under 10k iterations, so Ultralytics will pick AdamW automatically — but you should know MuSGD exists.

...

> **Practical takeaway:** YOLO26 isn't just a version bump. The small-object loss design and NMS-free inference are *direct* answers to two of the hardest things about document detection.

## 3. Transfer learning — why we don't train from scratch

`YOLO("yolo26n.pt")` loads weights pretrained on the **COCO** dataset (80 everyday object classes — people, cars, chairs). You might wonder: *"those classes have nothing to do with passports, why bother?"*

Because the early layers of any vision model learn **universal features**: edges, corners, textures, gradients, basic shapes. These don't care whether the input is a cat or a passport — they're useful for *any* image. By starting from COCO-trained weights, we get those universal features for free, and only need to teach the model the final layers — "what does *this specific class* look like".

### What transfer learning gives us

| Metric | From scratch | From pretrained |
|---|---|---|
| Epochs to converge | 200–500+ | 50–100 |
| Final mAP on small datasets | Often poor | Often surprisingly good |
| GPU hours needed | Many | Few |
| Risk of catastrophic overfitting on 50 images | High | Reduced |

For 50 images, training from scratch is essentially impossible — there isn't enough data to teach a network from zero. Transfer learning is what makes our project tractable at all.

> **Industry rule of thumb:** Always start from the largest available pretrained checkpoint that fits in memory. Train from scratch only if you have millions of images *and* a clear reason the pretrained model won't transfer.

## 4. Choosing a model size — n, s, m, l, x

YOLO26 comes in five sizes:

| Variant | Params | Use when |
|---|---|---|
| `yolo26n.pt` (nano) | ~3 M | Edge devices, real-time, **small datasets** ← us |
| `yolo26s.pt` (small) | ~12 M | Good balance, mobile-friendly |
| `yolo26m.pt` (medium) | ~25 M | Standard laptop GPU training |
| `yolo26l.pt` (large) | ~45 M | Server-side, accuracy-focused |
| `yolo26x.pt` (xlarge) | ~70 M | Highest accuracy, ignores compute cost |

### Why nano for our 50 images

Counterintuitively, a **smaller** model is often **better** when data is scarce:

- **Capacity matches data.** A 70M-parameter model with 50 training images will memorize every pixel and generalize to nothing. A 3M-param model can't memorize, so it's *forced* to learn patterns.
- **Faster iteration.** When you have to retrain 5 times to tune hyperparameters, nano takes minutes, x takes hours.
- **Easier deployment.** The eventual goal is to ship this in our backend; nano runs fast on CPU and is small enough to fit in any container.

> **Heuristic:** Match model capacity to dataset size. Hundreds of images → nano/small. Thousands → small/medium. Tens of thousands → medium/large. Millions → large/xlarge.

When you eventually grow the dataset to 500–1000 passports, switch to `yolo26s.pt` and retrain.

## 5. Every training argument decoded

Here's the call again, line by line:

```python
results = model.train(
    data="dataset/data.yaml",
    imgsz=640,
    batch=8,
    epochs=100,
    patience=20,
    augment=True,
    project=str(PROJECT_DIR),
    name="passport_v1",
)
```

### `data="dataset/data.yaml"`
The config file we wrote in notebook 2. YOLO reads it to find images, labels, and class names. **This is the only required argument.**

### `imgsz=640`
Input resolution — every image is letterboxed (resized + padded) to 640×640 before being fed to the model. Why 640?
- It's what YOLO26 was pretrained at on COCO. Using the same resolution preserves the pretrained features best.
- It's a power-of-2-multiple of the network's stride (32) — every layer's feature map dimensions are clean integers.
- It balances accuracy vs. speed. 1280 is more accurate but 4× slower; 320 is faster but loses small-object recall.

For our project, 640 is fine. If `dob` recall ends up poor and you suspect "the date is too small to detect", try `imgsz=1280` and compare.

### `batch=8`
Number of images processed together per gradient update.
- **Larger batch** → more stable gradients, but more GPU memory required, and *can* hurt generalization on small datasets.
- **Smaller batch** → noisier gradients (sometimes a *good* thing for regularization), less memory.

`batch=8` is conservative — fits in most laptop GPUs (2–4 GB VRAM), trains reasonably fast. If you have a beefy GPU, try `batch=16` or `batch=-1` (auto-determined to fit in memory).

### `epochs=100`
One epoch = one full pass through all 40 training images. So `epochs=100` = the model sees each training image 100 times.

For 50 images with augmentation, 100–300 epochs is typical. With `patience` (next), the actual training often stops earlier.

### `patience=20`
**Early stopping** — if val mAP doesn't improve for 20 consecutive epochs, training stops automatically. Why this matters:
- Saves compute on overfit territory (training continues, training loss keeps dropping, but val mAP plateaus or worsens — pure waste).
- Picks a good "stop point" without you having to babysit the run.
- The best checkpoint up to that point is what gets saved as `best.pt`.

`patience=20` on a 100-epoch budget is generous — gives the model room to escape plateaus before quitting.

### `augment=True`
**Important nuance:** in `model.train()`, the `augment` flag controls **test-time augmentation (TTA) on the validation pass**, not training augmentation. Training augmentations (mosaic, mixup, HSV jitter, flips, etc.) are *always on by default* and are controlled by separate hyperparameters (`mosaic`, `mixup`, `hsv_h`, `degrees`, etc.).

So `augment=True` here means: when validating, run inference at multiple scales/flips and average — slightly more accurate val metrics at the cost of slower validation. Useful when val set is small (it is, for us).

### `project=str(PROJECT_DIR)` and `name="passport_v1"`
Together these create the output directory: `<PROJECT_DIR>/<name>/` = `ai/runs/detect/passport_v1/`. Naming runs explicitly (instead of letting Ultralytics auto-name them `train`, `train2`, `train3`…) is critical when you have 20 experiments and need to know which is which. Tag runs by *what changed*: `passport_v1`, `passport_v2_imgsz1280`, `passport_v3_yolo26s`, etc.

## 6. Reading the training console output

While training runs, you'll see something like this scrolling past:

```
      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/100      1.2G      2.341      4.182      1.876         34        640
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)
                   all         10         50      0.421      0.389      0.234     0.087
```

Decode:

- `Epoch 1/100` — first of 100 epochs.
- `GPU_mem 1.2G` — VRAM in use. If this approaches your GPU's limit, lower `batch`.
- `box_loss 2.341` — how wrong the bounding-box coordinates are. Lower = better.
- `cls_loss 4.182` — how wrong the class predictions are.
- `dfl_loss 1.876` — distribution focal loss for box edge precision.
- `Instances 34` — total number of labeled boxes in this batch.
- `Size 640` — input resolution.

After each epoch, the validation pass prints a separate line:

- `Box(P)` = **Precision** of detected boxes
- `R` = **Recall** of detected boxes
- `mAP50` = mean Average Precision at 0.5 IoU threshold (the easier metric)
- `mAP50-95` = mean Average Precision averaged over IoU thresholds 0.5 to 0.95 in steps of 0.05 (the harder, more meaningful metric)

### What healthy training looks like

- All three losses (`box_loss`, `cls_loss`, `dfl_loss`) trend **downward** over epochs.
- `mAP50` and `mAP50-95` trend **upward**.
- Val and train losses move together. If train loss keeps dropping but val loss starts rising → **overfitting**.

### What unhealthy training looks like

| Symptom | Likely cause | Fix |
|---|---|---|
| All losses are NaN after epoch 1 | Bad data or learning rate too high | Verify labels are valid 0–1 range; reduce `lr0` |
| Loss flat from epoch 1 | Wrong `data.yaml` (no labels found) | Check `nc`, paths, folder structure |
| mAP stays at 0 | Class IDs don't match `nc` | Re-verify `data.yaml` ↔ label files |
| Train loss drops, val mAP plateaus low | Overfitting | More augmentation, smaller model, or more data |
| Train loss high and flat | Model too small or imgsz too low | Try `yolo26s.pt` or `imgsz=1280` |

## 7. Detection metrics — what Precision, Recall, and mAP actually mean

These are the numbers that tell you how good your model is. Knowing what they mean is the difference between "training looks good" and "I can defend this to my team."

### Precision and Recall (per class)

For each predicted box, IoU is computed against ground-truth boxes:

- **True Positive (TP)** — predicted box overlaps a ground-truth box of the same class with IoU ≥ threshold (e.g., 0.5)
- **False Positive (FP)** — predicted box has no matching ground truth (model hallucinated)
- **False Negative (FN)** — ground-truth box has no matching prediction (model missed it)

Then:

```
Precision = TP / (TP + FP)   "Of what I predicted, how much was right?"
Recall    = TP / (TP + FN)   "Of what existed, how much did I find?"
```

For passport extraction, **Recall matters more** — missing the passport number entirely is worse than predicting an extra spurious box you can filter later. For a fraud detector, **Precision matters more** — a false fraud accusation is costly.

### mAP — mean Average Precision

The detection-specific score that combines Precision and Recall across all classes and confidence thresholds:

- **AP** (Average Precision per class): area under the precision-recall curve as you sweep the confidence threshold.
- **mAP**: average of AP across all classes (the "m" = mean across classes).

Two common variants:
- **mAP50** — counts a prediction as correct if IoU ≥ 0.5. Easier metric. A good intuition baseline.
- **mAP50-95** — averages mAP at IoU thresholds 0.5, 0.55, 0.60, …, 0.95. Much harder. This is the **standard reported number** for YOLO benchmarks.

> **Reading mAP scores:**
> - `mAP50-95 < 0.30` — model is clearly underperforming, likely needs more data or different approach
> - `mAP50-95 = 0.30 – 0.50` — usable, with errors
> - `mAP50-95 > 0.50` — solid for production document AI
> - `mAP50-95 > 0.70` — excellent, especially for small objects
>
> **For comparison:** YOLO26n on COCO achieves ~40.9 mAP50-95 on 80 classes, 5000 val images. Our passport task is *much* simpler (1 or 5 classes, very consistent layout) — expect higher numbers, mid-0.6+ range with good annotation.

### Per-class metrics

The training output also shows per-class P, R, mAP. **Always look at these, not just the average.** The aggregate can hide that one class (often the smallest one — for us, `dob`) is underperforming. That's the class that needs more annotations or stronger augmentation.

## 8. The 50-image elephant in the room

50 images is a tiny dataset. We need to be honest about what that means and what helps.

### Why small datasets are hard

- A neural network with millions of parameters memorizes 50 images trivially.
- Variance in metrics is huge — change the random seed, get different numbers.
- Edge cases (rare passport years, heavy blur, glare) appear maybe 0–2 times. The model can't learn them.

### What we're doing about it

1. **Pretrained weights** — leverages COCO's millions of images for the base features.
2. **Nano model** — lower capacity, can't easily memorize.
3. **Heavy augmentation** — synthetically multiplies our 40 training images into thousands of unique training views (mosaic, mixup, HSV, rotation). Discussed below.
4. **Early stopping** — `patience=20` prevents us from overtraining into the overfit zone.
5. **Mosaic auto-disable** — Ultralytics turns off mosaic in the final epochs (`close_mosaic` parameter, default 10) so the model finishes training on real (un-mosaicked) images, matching deployment.

### What would help even more (not coded here)

| Action | Effort | Expected impact |
|---|---|---|
| Annotate 50 more passports | High | **Largest improvement possible.** Always the right answer when feasible. |
| Synthetic augmentation pipeline (Albumentations) | Medium | Helpful, especially for blur/glare/lighting variations |
| K-fold cross-validation | Low | Doesn't improve the model, but tells you a more honest accuracy number |
| Pseudo-labeling on unlabeled passports | Medium | Cheap if you have access to more passport images |

> **The hard truth:** for 50 images, the most impactful thing in this notebook is *not* tuning hyperparameters — it's recognizing when you've hit the data ceiling and going back to notebook 1 to label more.

## 9. Augmentation — how 40 images become 40,000 unique training views

When `model.train()` runs, every image is transformed each time it's seen. Default YOLO26 augmentations include:

| Augmentation | Default | What it does |
|---|---|---|
| **mosaic** | 1.0 | Combines 4 images into a 2×2 grid. The big one. |
| **mixup** | 0.0 (off for n) | Blends two images with their labels |
| **hsv_h** | 0.015 | Hue jitter |
| **hsv_s** | 0.7 | Saturation jitter |
| **hsv_v** | 0.4 | Brightness/value jitter |
| **degrees** | 0.0 | Rotation (small) |
| **translate** | 0.1 | Shift |
| **scale** | 0.5 | Zoom |
| **fliplr** | 0.5 | Horizontal flip (50%) |
| **close_mosaic** | 10 | Disable mosaic in last N epochs |

### Mosaic — why it's so important for small datasets

For each training step, mosaic picks 4 random images from the dataset, crops random regions of each, and stitches them into a single 2×2 collage. Labels are adjusted so all 4 sets of boxes are correctly placed in the new combined image.

Effect:
- Each "image" the model sees is essentially unique
- The model learns to detect objects in unusual contexts (a passport corner next to another passport corner)
- The effective dataset size multiplies — your 40 train images become a near-infinite stream of unique mosaics

Footnote from the docs: *"if the dataset is small, the same image may be used multiple times in the same mosaic"*. That's expected, not a bug.

### Why `close_mosaic` matters

The last 10 epochs disable mosaic. Reason: at deployment time, your model will see a single passport per image, not a 2×2 collage. Training those final epochs on real images "calibrates" the model to actual deployment conditions. Don't disable this.

### Should you change augmentation defaults?

**Usually no.** The Ultralytics defaults are tuned for typical object detection. Only change them with a specific reason:

- More blur in your real-world inputs? Add Albumentations blur in a custom dataloader.
- Flipping doesn't make sense (text is read left-to-right, flipping creates mirror-image text)? Set `fliplr=0` and `flipud=0`. **For passports, this is actually correct** — flipped passport text is meaningless. Consider:
  ```python
  results = model.train(..., fliplr=0.0, flipud=0.0)
  ```
- Want more rotation tolerance (for hand-held phone shots)? Try `degrees=10`.

## 10. Inspecting and validating the trained model

In [ ]:
# 10.1 — Confirm best.pt exists and load it
from pathlib import Path

best_pt = PROJECT_DIR / "passport_v1" / "weights" / "best.pt"
last_pt = PROJECT_DIR / "passport_v1" / "weights" / "last.pt"

print(f"best.pt exists: {best_pt.exists()}  ({best_pt})")
print(f"last.pt exists: {last_pt.exists()}  ({last_pt})")

if best_pt.exists():
    # Reload from the trained weights — this is the model you'd deploy
    trained = YOLO(str(best_pt))
    print(f"\nLoaded trained model. Task: {trained.task}, Classes: {trained.names}")

best.pt exists: True  (/Users/mac/Desktop/vision-project/ai/runs/detect/passport_v1/weights/best.pt)
last.pt exists: True  (/Users/mac/Desktop/vision-project/ai/runs/detect/passport_v1/weights/last.pt)

Loaded trained model. Task: detect, Classes: {0: 'personal_no'}


In [ ]:
# 10.2 — Run final validation explicitly. This regenerates val metrics from best.pt
# and saves a fresh confusion matrix and PR curves.
metrics = trained.val(data="dataset/data.yaml", imgsz=640, split="val")

print("\nFinal validation metrics:")
print(f"  mAP50:    {metrics.box.map50:.4f}")
print(f"  mAP50-95: {metrics.box.map:.4f}")
print(f"  Precision (mean): {metrics.box.mp:.4f}")
print(f"  Recall (mean):    {metrics.box.mr:.4f}")

print("\nPer-class mAP50-95:")
for cls_idx, cls_name in trained.names.items():
    if cls_idx < len(metrics.box.maps):
        print(f"  {cls_idx} {cls_name:>14s}: {metrics.box.maps[cls_idx]:.4f}")

Ultralytics 8.4.47 🚀 Python-3.11.14 torch-2.2.2 CPU (Intel Core i7-9750H 2.60GHz)
YOLO26n summary (fused): 122 layers, 2,375,031 parameters, 0 gradients, 5.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 824.7±268.3 MB/s, size: 68.3 KB)
val: Scanning /Users/mac/Desktop/vision-project/ai/notebooks/dataset/labels/val.cache... 10 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 10/10 2.5Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.1it/s 0.9s
                   all         10         10      0.965        0.9      0.986      0.753
Speed: 3.0ms preprocess, 77.6ms inference, 0.0ms loss, 0.2ms postprocess per image
Results saved to /Users/mac/Desktop/vision-project/ai/runs/detect/val

Final validation metrics:
  mAP50:    0.9859
  mAP50-95: 0.7528
  Precision (mean): 0.9645
  Recall (mean):    0.9000

Per-class mAP50-95:
  0    personal_no: 0.7528


In [ ]:
# 10.3 — Visual sanity check: predict on val images and look at the results
import matplotlib.pyplot as plt
from pathlib import Path

VAL_DIR = Path("dataset/images/val")
val_images = sorted(VAL_DIR.glob("*"))[:4]   # show first 4 val images

if not val_images:
    print("No val images found.")
else:
    preds = trained.predict(source=[str(p) for p in val_images], conf=0.25, save=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 14))
    for ax, p in zip(axes.flat, preds):
        # p.plot() returns a numpy array of the image with boxes drawn
        annotated = p.plot()
        # Ultralytics returns BGR; matplotlib expects RGB
        ax.imshow(annotated[..., ::-1])
        ax.set_title(Path(p.path).name)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


0: 480x640 2 personal_nos, 67.3ms
1: 480x640 1 personal_no, 67.3ms
2: 480x640 1 personal_no, 67.3ms
3: 480x640 1 personal_no, 67.3ms
Speed: 5.3ms preprocess, 67.3ms inference, 0.4ms postprocess per image at shape (1, 3, 480, 640)


<Figure size 1400x1400 with 4 Axes>

### What to look for in the visualizations

- **Boxes are tight around the actual fields** (not the printed labels, not the whole document)
- **Class labels are correct** (a "first_name" box is actually around the first name)
- **No missed fields** in clean images
- **Confidence scores are reasonable** (>0.5 for clear images)

If predictions are visibly wrong on multiple val images, the model isn't ready. Diagnostic order:
1. Re-run cell 10.2 — if `mAP50-95 < 0.3`, training didn't work
2. Check `results.png` in the run folder for loss curves
3. Check `confusion_matrix.png` for systematic class confusion
4. Check whether annotations were actually correct in notebook 1 (this is most often the real problem)

## 11. Industry-standard tips that will save you days

### 11.1 Name every run by what changed
Bad: `train`, `train2`, `train3`. Good: `passport_v1_baseline`, `passport_v2_imgsz1280`, `passport_v3_yolo26s_50epochs`. Future you in 2 weeks will not remember which was which without explicit names.

### 11.2 Log experiments with TensorBoard, Weights & Biases, or Comet
Ultralytics integrates natively. Add `tensorboard=True` (or use `--logger comet`) and you get a queryable history of every hyperparameter, every metric, every loss curve. This is how teams track 100+ experiments without losing their minds.

```python
results = model.train(..., plots=True)
# Then: tensorboard --logdir ai/runs/detect
```

### 11.3 Always save `args.yaml` with the weights
Ultralytics does this automatically, in the run folder. Don't delete it. Two months from now when someone asks "what hyperparameters produced this model?", you have an immediate answer.

### 11.4 Use `model.train(..., resume=True)` for crash recovery
If training crashes 70 epochs in (laptop sleeps, OOM, power outage), `resume=True` continues from the last checkpoint instead of restarting from epoch 0. Free recovery.

### 11.5 Test on a *holdout* set, not val
The val set has been peeked at every epoch — early stopping selected the best checkpoint based on it. Your reported metrics should ideally come from a separate test set the model has never influenced. We don't have one for our 50-image project, but for production datasets always carve one out.

### 11.6 Export for deployment, don't ship `.pt`
Once you have a model you like, export to a deployment format:
```python
trained.export(format="onnx")    # generic, runs in many runtimes
trained.export(format="openvino") # Intel CPU, very fast
trained.export(format="coreml")   # Apple devices
trained.export(format="tflite")   # mobile / edge
```
PyTorch `.pt` files require the full ultralytics dependency at inference time. ONNX/OpenVINO/CoreML are leaner, faster, and don't lock you into PyTorch.

### 11.7 Track inference latency, not just accuracy
A model with 0.95 mAP that takes 800ms per image is unshippable for many products. Run benchmark mode:
```python
trained.benchmark(data="dataset/data.yaml", imgsz=640)
```
Reports inference time across formats so you know what hardware your model needs.

### 11.8 Treat the first model as a lab rat, not a deliverable
The output of this notebook is *Model v1*. It exists to tell you what's wrong with your data and approach, not to ship. Plan for v2, v3, v4. Iteration is the actual job.

### 11.9 Common sources of "training looked great but production is bad"
- **Train/val too similar.** If you accidentally split such that train and val have the same passport copies, val mAP looks great and prod fails. The "group split" warning from notebook 2.
- **Distribution shift.** Trained on clean scans, deployed on phone photos. Add representative phone-quality samples to training data.
- **Class imbalance hidden by mAP.** Aggregate looks fine, but `dob` is at 0.2 and others at 0.9. Always check per-class numbers.
- **Different preprocessing in production than in training.** YOLO does specific letterboxing during training — replicate exactly the same preprocessing at inference, or use Ultralytics' `predict()` which handles it for you.

## 12. Summary

### What we did
1. Loaded YOLO26 nano with COCO-pretrained weights (transfer learning)
2. Pointed it at our `data.yaml`
3. Trained for up to 100 epochs with early stopping at patience=20
4. Saved outputs to `ai/runs/detect/passport_v1/`
5. Validated the resulting `best.pt` and visualized predictions

### What we learned
- **Why YOLO26**: NMS-free inference, ProgLoss/STAL for small objects, faster CPU performance
- **Why pretrained**: 50 images is too few to train from scratch — transfer learning is mandatory
- **Why nano**: small datasets need small models to avoid memorization
- **What every hyperparameter does**: imgsz, batch, epochs, patience, augment, project, name
- **How to read training output**: box_loss, cls_loss, dfl_loss, P, R, mAP50, mAP50-95
- **What the metrics actually mean**: precision vs recall, mAP@0.5 vs mAP@0.5–0.95
- **The 50-image reality**: augmentation helps but more data is the real fix
- **Industry practices**: naming runs, logging, exporting, benchmarking, holdout tests

### Coming next — `vision-v4.ipynb`

Connect detection to OCR:
- Run `best.pt` on a passport image to get field bounding boxes
- Crop each field
- Pass each crop through Tesseract / EasyOCR * / PaddleOCR
- Map crops to field names, return structured JSON
- Build a small FastAPI endpoint in `backend/` that wraps the whole pipeline

> **Reminder, again:** if your `mAP50-95` from this notebook is below ~0.40, **labeling 50 more images will improve the system more than anything in notebook 4 will.** Models love data more than they love clever code.